In [13]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from sklearn.metrics import classification_report


In [3]:
df_labels = pd.read_csv('labels.csv')

labele_map = dict(zip(df_labels['ClassId'] , df_labels['Name']))

In [16]:
IMG_SIZ = (64,64)
BATCH_SIZ = 32

#training
train_ds = tf.keras.utils.image_dataset_from_directory(
    'DATA',
    validation_split = 0.2,
    subset = "training",
    seed = 42,
    image_size = IMG_SIZ,
    batch_size = BATCH_SIZ
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'DATA',
    validation_split = 0.2,
    subset = "training",
    seed = 42,
    image_size = IMG_SIZ,
    batch_size = BATCH_SIZ
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    'TEST',
    image_size = IMG_SIZ,
    batch_size = BATCH_SIZ,
    shuffle = False
)

Found 5683 files belonging to 52 classes.
Using 4547 files for training.
Found 5683 files belonging to 52 classes.
Using 4547 files for training.
Found 433 files belonging to 52 classes.


In [17]:
num_classes = len(labele_map)

model = models.Sequential([
    layers.Rescaling(1./255 , input_shape=(64,64,3)),

    #feature extraction
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Classification Layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Helps prevent overfitting
    layers.Dense(num_classes, activation='softmax') # Softmax outputs probabilities
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Used because class targets are integers (0, 1, 2...)
    metrics=['accuracy']
)

model.summary()

c:\Users\eram\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 52)             │         6,708 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,242,804 (4.74 MB)

 Trainable params: 1,242,804 (4.74 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
test_loss , test_acc = model.evaluate(test_ds)

print (f"\nTest Accuracy: {test_acc * 100:2f}%")
print(f"Test Loss: {test_loss:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0162 - loss: 3.9436 

Test Accuracy: 1.616628%
Test Loss: 3.9436


In [20]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
predictions = model.predict(test_ds)
y_pred = np.argmax(predictions, axis=1)

target_names = [labele_map[i] for i in sorted(list(set(y_true)))]

print(classification_report(y_true, y_pred , target_names=target_names))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
                              precision    recall  f1-score   support

         Speed limit (5km/h)       0.00      0.00      0.00         4
        Speed limit (15km/h)       0.00      0.00      0.00         6
        Speed limit (30km/h)       0.00      0.00      0.00        12
        Speed limit (40km/h)       0.00      0.00      0.00        22
        Speed limit (50km/h)       0.00      0.00      0.00         9
        Speed limit (60km/h)       0.00      0.00      0.00        11
        Speed limit (70km/h)       0.00      0.00      0.00         2
        speed limit (80km/h)       0.00      0.00      0.00        15
    Dont Go straight or left       0.00      0.00      0.00        11
                    Unknown7       0.00      0.00      0.00        14
            Dont Go straight       0.01      0.10      0.02        10
                Dont Go Left       0.00      0.00      0.00        10
       Dont Go Left or Right       0.00      0.00

c:\Users\eram\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\eram\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\eram\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag